<a href="https://colab.research.google.com/github/Aravindmu/get-data-from-diff-sources-task3/blob/main/get_data_from_diff_sources_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [230]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

In [231]:
csv = pd.read_csv("population.csv")
csv = csv.rename(columns={'country': 'Country'})
csv.shape

(20, 2)

In [232]:
excel = pd.read_excel("gdp.xlsx")
excel.shape

(20, 2)

In [233]:
json=pd.read_json("internet_users.json")
json = json.rename(columns={'country': 'Country'})
json.shape

(20, 2)

In [234]:
tree = ET.parse("literacy_rate.xml")
root = tree.getroot()
xml_list = []
for row in root:
    # Try to find 'name' tag for the country
    country = row.find('name').text if row.find('name') is not None else None
    literacy_rate = row.find('literacy_rate').text if row.find('literacy_rate') is not None else None
    xml_list.append({"Country": country, "literacy_rate": literacy_rate})
xml = pd.DataFrame(xml_list)

xml.shape

(20, 2)

In [235]:
for df in [csv, excel, json, xml]:
    if "Country" in df.columns:
        df["Country"] = df["Country"].astype(str).str.strip().str.title()

In [236]:
csv["population"] = pd.to_numeric(
    csv["population"].astype(str).str.replace(",", ""), errors="coerce"
)
csv["population"] = csv["population"].fillna(
    csv["population"].median()
)

In [237]:
excel["GDP"] = pd.to_numeric(excel["GDP"], errors="coerce")
excel["GDP"] = excel["GDP"].fillna(0)

In [238]:
xml["literacy_rate"] = pd.to_numeric(
    xml["literacy_rate"], errors="coerce"
)

xml

,Country,literacy_rate
0,None,72.7
1,None,82.2
2,None,80.3
3,None,97.5
4,None,92.9
5,None,89.1
6,None,81.0
7,None,82.9
8,None,97.6
9,None,83.7


In [239]:
merged = csv.merge(excel, on="Country", how="outer")
merged = merged.merge(json, on="Country", how="outer")
merged = merged.merge(xml, on="Country", how="outer")
display(merged.head())

,Country,population,GDP,internet_users,literacy_rate
0,Argentina,1.432830e+09,8.767337e+12,421477197.0,NaN
1,Australia,4.392857e+08,5.444268e+12,201619113.0,NaN
2,Brazil,6.750950e+08,8.793053e+12,24027075.0,NaN
3,Canada,1.438268e+09,8.158352e+12,894102645.0,NaN
4,China,7.928464e+08,5.086548e+12,685275917.0,NaN


In [240]:
merged["Internet Penetration Rate"] = merged.apply(
    lambda row: (
        (row["internet_users"] / row["population"]) * 100
        if row["population"] > 0
        else 0
    ),
    axis=1,
).round(2)

In [241]:
merged.columns.tolist()

['Country',
 'population',
 'GDP',
 'internet_users',
 'literacy_rate',
 'Internet Penetration Rate']

In [242]:
merged.head()

,Country,population,GDP,internet_users,literacy_rate,Internet Penetration Rate
0,Argentina,1.432830e+09,8.767337e+12,421477197.0,NaN,29.42
1,Australia,4.392857e+08,5.444268e+12,201619113.0,NaN,45.90
2,Brazil,6.750950e+08,8.793053e+12,24027075.0,NaN,3.56
3,Canada,1.438268e+09,8.158352e+12,894102645.0,NaN,62.17
4,China,7.928464e+08,5.086548e+12,685275917.0,NaN,86.43


In [243]:
highest_internet_penetration = merged.sort_values(by='Internet Penetration Rate', ascending=False).head(5)
display(highest_internet_penetration[['Country', 'Internet Penetration Rate']])

,Country,Internet Penetration Rate
10,Japan,92.22
36,Spain,88.46
4,China,86.43
34,South Africa,80.61
3,Canada,62.17


In [244]:
average_literacy_rate = merged['literacy_rate'].mean()
print(f"Average Literacy Rate: {average_literacy_rate:.2f}%")

Average Literacy Rate: 78.43%


In [245]:
analysis_columns = ['GDP', 'population', 'literacy_rate', 'Internet Penetration Rate']
correlation_matrix = merged[analysis_columns].dropna().corr()
display(correlation_matrix)

,GDP,population,literacy_rate,Internet Penetration Rate
GDP,NaN,NaN,NaN,NaN
population,NaN,NaN,NaN,NaN
literacy_rate,NaN,NaN,NaN,NaN
Internet Penetration Rate,NaN,NaN,NaN,NaN


### Load into SQLite

In [246]:
import sqlite3
conn = sqlite3.connect('memory')

merged.to_sql('data', conn, if_exists='replace', index=False)

print("Merged DataFrame loaded into SQLite table 'data'.")

Merged DataFrame loaded into SQLite table 'data'.


In [247]:
query_a = """
SELECT Country, "Internet Penetration Rate" FROM data
ORDER BY "Internet Penetration Rate" DESC
LIMIT 5;
"""
highest_internet_penetration_sql = pd.read_sql_query(query_a, conn)
display(highest_internet_penetration_sql)

,Country,Internet Penetration Rate
0,Japan,92.22
1,Spain,88.46
2,China,86.43
3,South Africa,80.61
4,Canada,62.17


In [248]:
query_b = """
SELECT AVG(literacy_rate) FROM data;
"""
average_literacy_rate_sql = pd.read_sql_query(query_b, conn)
print(f"Average Literacy Rate: {average_literacy_rate_sql.iloc[0,0]:.2f}%")

Average Literacy Rate: 78.43%


In [249]:
query_c = """
SELECT literacy_rate, "Internet Penetration Rate" FROM data
WHERE literacy_rate IS NOT NULL AND "Internet Penetration Rate" IS NOT NULL;
"""
correlation_data = pd.read_sql_query(query_c, conn)
correlation = correlation_data['literacy_rate'].corr(correlation_data['Internet Penetration Rate'])
print(f"Correlation between Literacy Rate and Internet Penetration: {correlation:.2f}")

Correlation between Literacy Rate and Internet Penetration: nan


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [250]:
query_d = """
SELECT Country, GDP, population, (GDP * 1.0 / population) AS "GDP Per Capita" FROM data
WHERE population IS NOT NULL AND population > 0;
"""
gdp_per_capita_analysis = pd.read_sql_query(query_d, conn)
display(gdp_per_capita_analysis.head())

,Country,GDP,population,GDP Per Capita
0,Argentina,8.767337e+12,1.432830e+09,6118.894099
1,Australia,5.444268e+12,4.392857e+08,12393.456887
2,Brazil,8.793053e+12,6.750950e+08,13024.912742
3,Canada,8.158352e+12,1.438268e+09,5672.346150
4,China,5.086548e+12,7.928464e+08,6415.552759


In [251]:
query_e = """
SELECT Country, (GDP * 1.0 / population) AS "GDP Per Capita" FROM data
WHERE population IS NOT NULL AND population > 0 AND (GDP * 1.0 / population) > 10000
ORDER BY "GDP Per Capita" DESC;
"""
gdp_above_10k = pd.read_sql_query(query_e, conn)
display(gdp_above_10k)

,Country,GDP Per Capita
0,Indonesia,132792.830386
1,Italy,50722.583577
2,Russia,26099.651924
3,France,17861.109518
4,United Kingdom,17481.480607
5,Spain,16998.007574
6,India,14520.297092
7,Brazil,13024.912742
8,Australia,12393.456887
9,South Korea,10609.152180


In [252]:
query_f = """
SELECT SUM(population) FROM data;
"""
total_population = pd.read_sql_query(query_f, conn)
print(f"Total population covered: {total_population.iloc[0,0]:,.0f}")

Total population covered: 14,864,685,089


In [253]:
query_g = """
SELECT Country, literacy_rate, "Internet Penetration Rate" FROM data
WHERE literacy_rate IS NOT NULL
ORDER BY literacy_rate ASC
LIMIT 5;
"""
lowest_literacy_rates = pd.read_sql_query(query_g, conn)
display(lowest_literacy_rates)

,Country,literacy_rate,Internet Penetration Rate
0,None,60.5,0.0
1,None,60.6,0.0
2,None,66.4,0.0
3,None,67.8,0.0
4,None,70.8,0.0


In [254]:
query_h = """
SELECT Country, GDP, population FROM data
ORDER BY GDP DESC
LIMIT 5;
"""
top_5_wealthiest = pd.read_sql_query(query_h, conn)
display(top_5_wealthiest)

,Country,GDP,population
0,Spain,1.527551e+13,8.986649e+08
1,India,1.454072e+13,1.001406e+09
2,Italy,1.290723e+13,2.544672e+08
3,Japan,1.247384e+13,1.206264e+09
4,Indonesia,1.240414e+13,9.340975e+07


In [255]:
query_i = """
SELECT Country, "Internet Penetration Rate" FROM data
WHERE "Internet Penetration Rate" > 70
ORDER BY "Internet Penetration Rate" DESC;
"""
internet_over_70_percent = pd.read_sql_query(query_i, conn)
display(internet_over_70_percent)

,Country,Internet Penetration Rate
0,Japan,92.22
1,Spain,88.46
2,China,86.43
3,South Africa,80.61


In [256]:
query_j = """
SELECT AVG(GDP * 1.0 / population) FROM data
WHERE "Internet Penetration Rate" > 50 AND population IS NOT NULL AND population > 0;
"""
average_gdp_per_capita_high_internet = pd.read_sql_query(query_j, conn)
print(f"Average GDP Per Capita for countries with >50% Internet Penetration: ${average_gdp_per_capita_high_internet.iloc[0,0]:,.2f}")

Average GDP Per Capita for countries with >50% Internet Penetration: $8,993.62


In [257]:
query_k = """
SELECT COUNT(Country) AS NumCountries, AVG("Internet Penetration Rate") AS AverageInternetPenetration
FROM data
WHERE literacy_rate > 90;
"""
high_literacy_insights = pd.read_sql_query(query_k, conn)
display(high_literacy_insights)

,NumCountries,AverageInternetPenetration
0,3,0.0
